# PID tuning

A first-order process with dead time, `K = 2`, `τ = 20 s`, `θ = 4 s`, is
controlled by a PID (derivative on the measurement, anti-windup). The setpoint
steps from 20 to 50 at t = 10 s. **Run all the cells**, then turn the knobs: the
closed-loop response and the indicators are recomputed by the knob callbacks.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import numpy as np

import anywidget_instruments as ai


def closed_loop(kp, ti, td, gain=2.0, tau=20.0, delay=4.0, dt=0.2, duration=200.0):
    """SP, PV and OP of the loop, one row per sample."""
    pid = ai.PID(kp=kp, ti=ti, td=td, sp=20.0, output=10.0)
    pipe = [10.0] * int(delay / dt)  # dead time
    y = 20.0
    rows = []
    for k in range(int(duration / dt)):
        pid.sp = 50.0 if k * dt >= 10.0 else 20.0
        u = pid.step(y, dt)
        pipe.append(u)
        y += dt / tau * (-y + gain * pipe.pop(0))
        rows.append((pid.sp, y, u))
    return np.array(rows)


def metrics(rows, dt=0.2):
    pv = rows[int(10.0 / dt) :, 1]
    overshoot = max(0.0, (pv.max() - 50.0) / 30.0 * 100.0)
    outside = np.nonzero(np.abs(pv - 50.0) > 0.02 * 30.0)[0]
    settling = (outside[-1] + 1) * dt if len(outside) else 0.0
    return overshoot, settling

In [ ]:
ai.theme_switch()  # light / dark widgets (STYLE-007)

In [ ]:
kp = ai.Knob(1.2, min=0, max=5, step=0.05, label="Kp", size=(120, 120))
ti = ai.Knob(20, min=1, max=100, step=1, unit="s", label="Ti", size=(120, 120))
td = ai.Knob(0, min=0, max=20, step=0.5, unit="s", label="Td", size=(120, 120))
chart = ai.WaveformChart(
    n_traces=3,
    history=1000,
    dt=0.2,
    x_unit="s",
    y_min=0,
    y_max=100,
    traces=[{"name": "SP"}, {"name": "PV"}, {"name": "OP"}],
    label="Step response",
    size=(620, 240),
)
overshoot = ai.AnalogIndicator(max=50, unit="%", normal_hi=10, hi=20, label="Overshoot")
settling = ai.AnalogIndicator(max=190, unit="s", normal_hi=60, hi=120, label="Settling time (2 %)")


def update(_change=None):
    rows = closed_loop(kp.value, ti.value, td.value)
    chart.clear()
    chart.append(rows)
    overshoot.value, settling.value = metrics(rows)


for knob in (kp, ti, td):
    knob.on_change(update)
update()

ai.Panel([kp, ti, td, chart, overshoot, settling], columns=3)

**Things to try**: raise Kp until the response oscillates, then add some
derivative action (Td) to damp it; a very short Ti gives a large overshoot.